In [0]:
from pyspark.sql import functions as F

data = [
    ("A", "2026-04-20 10:00:10", 25.1),
    ("A", "2026-04-20 10:01:20", 25.4),
    ("A", "2026-04-20 10:04:50", 25.7),
    ("A", "2026-04-20 10:02:10", 25.3),  # 模擬晚到事件
    ("A", "2026-04-20 10:11:00", 26.1),
]

df = spark.createDataFrame(data, ["device_id", "event_time", "temperature"]) \
    .withColumn("event_time", F.to_timestamp("event_time"))

display(df)


result_df = (
    df.groupBy(
        F.window("event_time", "5 minutes"),
        F.col("device_id")
    )
    .agg(
        F.avg("temperature").alias("avg_temp"),
        F.count("*").alias("cnt")
    )
    .orderBy("window")
)

display(result_df)


df_with_processing = (
    df.withColumn("processing_time", F.to_timestamp(F.lit("2026-04-20 10:12:00")))
      .withColumn(
          "delay_minutes",
          (F.col("processing_time").cast("long") - F.col("event_time").cast("long")) / 60
      )
)

display(df_with_processing)